# 실험 08 — 환자별 변이 부담을 반영한 특징 표현 비교

**연구 질문:** 전체 환자에게 동일한 유전자 0/1 표현을 적용하는 것보다, 환자별 변이 수 및 변이 부담 구간별 유전자 정보를 반영하면 분류 오류가 줄어드는가?

- **Baseline:** 유전자 변이 유무 4,384개 + SGD
- **Count:** 기존 특징 + 환자별 log(1+변이 개수) (Fold 학습부만 표준화)
- **Interaction:** 기존 특징 + 변이 개수 정보 + 저변이/고변이 구간에서만 활성화되는 유전자별 상호작용 특징
- 각 방법은 같은 SGD 설정과 **3-Fold StratifiedGroupKFold**로 평가
- 구간 경계(25/75백분위), 스케일러는 반드시 각 Fold 학습 데이터에서만 계산
- 소수 암종과 주요 혼동 암종, 환자군별 오류율도 함께 비교
- 본 실험은 검증 예측 결과를 본 뒤 구상한 **탐색적 개선 실험**임. 독립 검증이나 nested CV 없이는 최종 일반화 성능을 입증할 수 없음
- **공식 Test 데이터는 읽지 않음**


In [ ]:
# 필요한 패키지 (최초 1회만 VS Code 터미널에서 실행)
# python -m pip install numpy pandas scipy scikit-learn matplotlib
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from IPython.display import display

SEED = 42
N_SPLITS = 3
QUICK_CHECK = False  # True면 첫 번째 Fold만 확인; 정식 분석은 False
ROOT = Path.cwd()
DATA_DIR = ROOT / 'outputs'
OUT = ROOT / 'experiment_08'
OUT.mkdir(exist_ok=True)
assert (DATA_DIR/'X_train_binary.npz').exists(), 'outputs/X_train_binary.npz가 없습니다.'
print('프로젝트:', ROOT.resolve())

In [ ]:
X = sparse.load_npz(DATA_DIR/'X_train_binary.npz').tocsr().astype(np.float32)
X.sum_duplicates(); X.eliminate_zeros(); X.sort_indices()
y_text=np.load(DATA_DIR/'y_train.npy',allow_pickle=True).astype(str)
assert X.shape==(6201,4384) and len(y_text)==X.shape[0], '입력 데이터 순서/형태를 확인하세요.'
assert np.isin(X.data,[0,1]).all(), '0/1 이진 특징이 아닙니다.'
le=LabelEncoder().fit(y_text)
y=le.transform(y_text)
classes=list(le.classes_)
burden=np.diff(X.indptr).astype(np.float32)
# 중복 이진 프로파일은 모두 같은 검증 그룹 (0/1만 포함하므로 nonzero index 기준으로 해시)
profiles=[hashlib.sha256(X.indices[X.indptr[i]:X.indptr[i+1]].tobytes()).digest() for i in range(X.shape[0])]
groups=pd.factorize(np.array(profiles,dtype='|S32'))[0]
folds=list(StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED).split(X,y,groups))
for f,(tr,va) in enumerate(folds,1):
    assert set(groups[tr]).isdisjoint(groups[va])
    assert set(np.unique(y[tr]))==set(range(len(classes)))
    print(f'Fold {f}: train={len(tr)}, valid={len(va)}, 검증 클래스={len(np.unique(y[va]))}')
if QUICK_CHECK: folds=folds[:1]
print('Train 샘플:',len(y),'클래스:',len(classes),'프로파일 그룹:',len(np.unique(groups)))

## 특징 설계

변이 수가 적거나 많은 환자에게 **서로 다른 유전자-변이수 결합 특징**을 제공하되, 임의로 환자군에 정답 암종이나 오류 여부를 부여하지 않는다. 

**주요 통제:** 변이 수 구간 임계값 및 변이 수 스케일러를 `fit`하는 과정에는 각 Fold의 학습 데이터만 활용한다.

In [ ]:
def make_features(X_tr, X_va, strategy):
    # X_tr, X_va는 이미 같은 유전자 순서의 CSR 희소행렬
    if strategy == 'Binary':
        return X_tr, X_va, {'feature_columns': X_tr.shape[1]}
    count_tr=np.diff(X_tr.indptr).astype(np.float32)
    count_va=np.diff(X_va.indptr).astype(np.float32)
    scaler=StandardScaler()
    c_tr=scaler.fit_transform(np.log1p(count_tr).reshape(-1,1)).astype(np.float32)
    c_va=scaler.transform(np.log1p(count_va).reshape(-1,1)).astype(np.float32)
    tr_parts=[X_tr,sparse.csr_matrix(c_tr)]
    va_parts=[X_va,sparse.csr_matrix(c_va)]
    meta={'feature_columns':X_tr.shape[1]+1,'count_mean_train':float(scaler.mean_[0])}
    if strategy == 'Interaction':
        q25,q75=np.percentile(count_tr,[25,75]) # train fold only
        for predicate in [lambda a:a<=q25,lambda a:a>=q75]:
            mask_tr=predicate(count_tr).astype(np.float32)
            mask_va=predicate(count_va).astype(np.float32)
            tr_parts.append(X_tr.multiply(mask_tr[:,None]).tocsr())
            va_parts.append(X_va.multiply(mask_va[:,None]).tocsr())
        meta.update({'q25_train':float(q25),'q75_train':float(q75),'feature_columns':3*X_tr.shape[1]+1})
    return sparse.hstack(tr_parts,format='csr'),sparse.hstack(va_parts,format='csr'),meta

STRATEGIES=['Binary','Count','Interaction']

def build_model():
    return SGDClassifier(loss='log_loss',alpha=1e-4,class_weight='balanced',
                         max_iter=1500,tol=1e-3,random_state=SEED)


In [ ]:
metric_rows=[]; oof_rows=[]; fold_meta=[]
for fold,(tr,va) in enumerate(folds,1):
    print(f'\nFold {fold}/{len(folds)}')
    for strategy in STRATEGIES:
        Xt,Xv,meta=make_features(X[tr],X[va],strategy)
        model=build_model(); model.fit(Xt,y[tr]); pred=model.predict(Xv)
        macro=f1_score(y[va],pred,labels=np.arange(len(classes)),average='macro',zero_division=0)
        acc=accuracy_score(y[va],pred)
        metric_rows.append({'fold':fold,'strategy':strategy,'macro_f1':macro,'accuracy':acc,'n_valid':len(va)})
        fold_meta.append({'fold':fold,'strategy':strategy,**meta})
        for j,p in zip(va,pred):
            oof_rows.append({'row_index':int(j),'fold':fold,'strategy':strategy,
                             'true':str(y_text[j]),'pred':str(classes[p]),
                             'correct':int(p==y[j]),'mutation_count':int(burden[j])})
        print(f'  {strategy:12s} Macro F1={macro:.4f}, Accuracy={acc:.4f}, features={Xt.shape[1]:,}')
metrics=pd.DataFrame(metric_rows)
oof=pd.DataFrame(oof_rows)
summary=metrics.groupby('strategy',as_index=False).agg(macro_f1_mean=('macro_f1','mean'),macro_f1_std=('macro_f1','std'),accuracy_mean=('accuracy','mean'))
display(summary.sort_values('macro_f1_mean',ascending=False))

## 오류 분석

- **암종별 F1:** OOF 전체 환자 기준 (모든 Fold를 실행한 경우에만)
- **소수 클래스:** 전체 Train에서 환자 수가 적은 클래스 20%에 해당하는 하위 6개 암종 (분류 방식은 사전 정의; 분할별 학습에 레이블 빈도를 재활용하지 않음)
- **변이 수 구간:** 0, 1~7, 8~27, 28개 이상 (탐색적 기술통계용 고정 구간)
- **공통 오분류 대상:** 실험 07에서 탐색된 환자의 정답 정보를 이번 모델의 학습이나 선택에 사용하지 않음.

In [ ]:
assert not oof.duplicated(['row_index','strategy']).any()
class_rows=[]; pair_rows=[]; burden_rows=[]
PAIRS=[('KIPAN','KIRC'),('GBMLGG','LGG'),('BRCA','OV')]
minority=list(pd.Series(y_text).value_counts().sort_values().index[:6])
for strategy in STRATEGIES:
    d=oof[oof.strategy==strategy].copy()
    for lab in classes:
        yb=(d.true==lab).astype(int)
        pb=(d.pred==lab).astype(int)
        class_rows.append({'strategy':strategy,'cancer':lab,'n':int(yb.sum()),
                           'f1':f1_score(yb,pb,zero_division=0)})
    for a,b in PAIRS:
        pair_rows.append({'strategy':strategy,'pair':f'{a} <-> {b}',
                          'errors':int(((d.true==a)&(d.pred==b)).sum()+((d.true==b)&(d.pred==a)).sum())})
    d['burden_group']=pd.cut(d.mutation_count,bins=[-1,0,7,27,np.inf],labels=['0개','1~7개','8~27개','28개 이상'])
    for name,g in d.groupby('burden_group',observed=True):
        burden_rows.append({'strategy':strategy,'burden_group':str(name),
                            'n':len(g),'error_rate':float(1-g.correct.mean())})
class_df=pd.DataFrame(class_rows)
pair_df=pd.DataFrame(pair_rows)
burden_df=pd.DataFrame(burden_rows)
minority_df=class_df[class_df.cancer.isin(minority)].groupby('strategy',as_index=False).agg(minority_macro_f1=('f1','mean'))
display(minority_df)
display(pair_df.pivot(index='pair',columns='strategy',values='errors'))
display(burden_df.pivot(index='burden_group',columns='strategy',values='error_rate'))
if not QUICK_CHECK:
    pivot=class_df.pivot(index='cancer',columns='strategy',values='f1')
    pivot['interaction_minus_binary']=pivot['Interaction']-pivot['Binary']
    display(pivot.sort_values('interaction_minus_binary',ascending=False).round(3))

## 결과 저장 및 주의

**결과 해석 시 주의:** 실험 07에서 발견한 환자군을 기준으로 실험 08의 아이디어를 제안했기 때문에, 같은 Train 데이터의 OOF 점수를 여러 차례 참고한 선택 편향이 남아 있을 수 있다. 최종 모델 선택에는 별도의 바깥 검증 또는 새로 고정된 평가 분할이 권장된다. 실험 08에서 좋아진 모델을 그대로 공식 Test에 맞춰 수정하지 않는다.

In [ ]:
metrics.to_csv(OUT/'cv_metrics.csv',index=False,encoding='utf-8-sig')
summary.to_csv(OUT/'cv_summary.csv',index=False,encoding='utf-8-sig')
oof.to_csv(OUT/'oof_predictions.csv',index=False,encoding='utf-8-sig')
class_df.to_csv(OUT/'class_f1.csv',index=False,encoding='utf-8-sig')
pair_df.to_csv(OUT/'pair_confusion.csv',index=False,encoding='utf-8-sig')
burden_df.to_csv(OUT/'burden_errors.csv',index=False,encoding='utf-8-sig')
minority_df.to_csv(OUT/'minority_f1.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(fold_meta).to_csv(OUT/'fold_feature_metadata.csv',index=False,encoding='utf-8-sig')
(OUT/'config.json').write_text(json.dumps({'seed':SEED,'n_splits':N_SPLITS,'quick_check':QUICK_CHECK,'strategies':STRATEGIES,'minority_labels':minority,'test_used':False,'validation':'StratifiedGroupKFold grouped binary profiles'},ensure_ascii=False,indent=2),encoding='utf-8')
print('저장 완료:',OUT.resolve())
print('실험 08 폴더를 ZIP으로 압축해 업로드하면 분석을 이어갈 수 있습니다.')